## Step 1 - Preprocessing, RMSD & RMSF, pocket detection, separation & characterisation
This is the first explanatory notebook of the pipeline.
The wrapped trajectories and topologies under input/ are processed (drying the protein, aligning it and running the pocket detection steps) to produce the output used in the remaining pipeline.
The workflow is based on MDanalysis and mdpocket (fpocket) and can be customised following their documentation.

Of course, we will start with all necessary imports and useful settings, before assembling the pipeline.

In [ ]:
import os, sys
import importlib
import time
import warnings
import scripts.preprocessing as prepro
import scripts.basic_analysis as basic_ana
import scripts.pocket_analysis as pocket_ana
import scripts.gene_selections as gene_sel
import scripts.logging as logger
import config as conf

importlib.reload(prepro)
importlib.reload(basic_ana)
importlib.reload(pocket_ana)
importlib.reload(gene_sel)
importlib.reload(logger)
importlib.reload(conf)

warnings.filterwarnings("ignore")

Define the global variables

In [ ]:
# Globals
VERBOSE = True
SHOW_PLOTS = False
APO_STRUCTURES_DIR = conf.APO_STRUCTURES_DIR
HOLO_STRUCTURES_DIR = conf.HOLO_STRUCTURES_DIR

# Simulation timing: frames are written every traj_period integration steps of time_step fs, then strided by `stride` when saving the dry protein.
# In our case the input trajectories have been strided to achieve 1 frame = 1 ns
TRJ_PERIOD_STEP_STRIDE = {'traj_period': 25000, 'time_step': 4, 'stride': 1}
STRIDE = TRJ_PERIOD_STEP_STRIDE['stride']

# Binding site & ICL3 residue numbers per gene (hTAAR1, mTAAR1, mTAAR7f, mTAAR9) are read from reference_data/binding_site_residues.txt and
# reference_data/ICL3_definition.txt by scripts/gene_selections.py

DEV_SUBSET = {'8ITF': ['1', '2']} # TODO: delete once the notebooks / code are clean

Next, we have some helper functions defining our project list and setting up the results folders. `get_project_list()` respects `DEV_SUBSET` from the Globals cell above -- set it to `None` there to run the full dataset instead.

In [ ]:
def get_project_list(subset=DEV_SUBSET):
    """[<APO_STRUCTURES_DIR or HOLO_STRUCTURES_DIR>/<state><PDBID>/<rep>, ...] for every
    replicate under both input structure trees,
    e.g., '.../input/apo_structures/apo8ITF/1'

    subset: optional {pdb_id: [reps]} filter (bare PDB ID, no apo/holo prefix -- applies to
    both states), e.g. {'8ITF': ['1', '2']}. None returns every replicate found on disk."""
    project_ls = []
    for structures_dir in (APO_STRUCTURES_DIR, HOLO_STRUCTURES_DIR):
        if not os.path.isdir(structures_dir):
            continue
        for folder in sorted(os.listdir(structures_dir)):
            folder_path = os.path.join(structures_dir, folder)
            if not os.path.isdir(folder_path):
                continue
            pdb_id = folder[3:] if folder.startswith('apo') else folder[4:]
            if subset is not None and pdb_id not in subset:
                continue
            for rep in sorted(os.listdir(folder_path)):
                rep_path = os.path.join(folder_path, rep)
                if not os.path.isdir(rep_path):
                    continue
                if subset is not None and rep not in subset[pdb_id]:
                    continue
                project_ls.append(rep_path)
    return project_ls


def setup_results_folder(project_ls):
    """Mirrors project_ls's PDB+rep entries into the matching output results tree
    (output/apo_structures/ or output/holo_structures/, see config.results_dir_for) -- takes
    project_ls directly (rather than re-walking input/ itself) so the output tree it creates
    always matches exactly what get_project_list() selected, subset included."""
    for proj_rep in project_ls:
        curr_rep = os.path.basename(proj_rep)
        folder_name = os.path.basename(os.path.dirname(proj_rep))
        results_dir = conf.results_dir_for(folder_name)
        os.makedirs(os.path.join(results_dir, folder_name, curr_rep), exist_ok=True)


project_ls = get_project_list()
project_ls

## Setting up the results folders
Before running any analysis, we mirror the input PDB+rep tree into the output folder, so that every project and replicate has a dedicated results folder to save into.

In [ ]:
setup_results_folder(project_ls)

## Step 1.1 - Preprocessing: drying the protein & aligning to the reference structure
For every project and replicate, we first dry the protein (i.e., select only the protein atoms, dropping solvent, ions, membrane, etc.) and save it, then align the resulting trajectory to the reference topology used as simulation input. This is handled by `PreProcess` in `scripts/preprocessing.py`. Bear in mind that the trajectories and topolgies need to follow the name conventions, i.e., structure.pdb and traj_wrapped.xtc. Here, the trajectory can also be strided, e.g., by 10.

In [ ]:
for proj_rep in project_ls:
    curr_rep = os.path.basename(proj_rep)
    folder_name = os.path.basename(os.path.dirname(proj_rep))
    curr_traj = os.path.join(proj_rep, 'traj_wrapped.xtc')
    curr_topol = os.path.join(proj_rep, 'structure.pdb')
    results_dir = conf.results_dir_for(folder_name)

    # I: dry the protein and save it
    preprocess_obj = prepro.PreProcess(curr_topol, curr_traj, folder_name, curr_rep, verbose=VERBOSE)
    preprocess_obj.make_selection_n_save(stride=STRIDE)  # if selection None selects 'protein'

    # II: align to reference (dry protein is saved first, align_to_reference takes the path to it --
    # otherwise issues with the number of frames: only 1st frame is returned as AtomGroup)
    dry_top_path = os.path.join(results_dir, folder_name, curr_rep, 'dry_prot.pdb')
    dry_traj_path = os.path.join(results_dir, folder_name, curr_rep, 'dry_prot.xtc')
    preprocess_obj.align_to_reference(reference=curr_topol, universe=(dry_top_path, dry_traj_path))

## Step 1.2 - RMSD & RMSF
With the aligned trajectories in place, we calculate the RMSD (overall Cα, Cα without ICL3, and the binding site) and the per-residue RMSF for every project and replicate, using `BasicAnalysis` in `scripts/basic_analysis.py`. The binding site and ICL3 residue numbers differ per gene (hTAAR1, mTAAR1, mTAAR7f, mTAAR9), so the group selection is resolved per project via `scripts/gene_selections.py`.

In [ ]:
for proj_rep in project_ls:
    curr_rep = os.path.basename(proj_rep)
    folder_name = os.path.basename(os.path.dirname(proj_rep))
    curr_traj = os.path.join(proj_rep, 'traj_wrapped.xtc')
    curr_topol = os.path.join(proj_rep, 'structure.pdb')

    # binding site & ICL3 residues differ per gene, so the group selection is built per project
    gene = gene_sel.gene_for_project(folder_name)
    group_sel = gene_sel.group_selection_for(gene)

    basic_analysis_obj = basic_ana.BasicAnalysis(curr_topol, curr_traj, curr_proj=folder_name,
                                                 curr_rep=curr_rep, verbose=VERBOSE, show_plots=SHOW_PLOTS)
    basic_analysis_obj.calc_rmsd(trj_period_step_stride=TRJ_PERIOD_STEP_STRIDE,
                                 selection='protein and name CA', group_selection=group_sel,
                                 icl3_free_selection=gene_sel.ca_without_icl3_selection_for(gene))
    basic_analysis_obj.calc_rmsf()

## Step 1.3 - Pocket detection with MDpocket
Finally, we run the MDpocket-based pocket search on every project and replicate, using `PocketAnalysis` in `scripts/pocket_analysis.py`. This runs on the aligned, dry-protein trajectory/topology written by Step 1.1 (not the raw input, which still has solvent/membrane) -- resolved per project via `conf.aligned_paths_for`. The pocket density grids are saved into the `pockets` subfolder and are the input for Step 2 of the pipeline.

In [ ]:
for proj_rep in project_ls:
    curr_rep = os.path.basename(proj_rep)
    folder_name = os.path.basename(os.path.dirname(proj_rep))
    curr_topol, curr_traj = conf.aligned_paths_for(folder_name, curr_rep, 'xtc')
    pocket_obj = pocket_ana.PocketAnalysis(curr_topol, curr_traj, curr_proj=folder_name, curr_rep=curr_rep,
                                           verbose=VERBOSE, pocket_dir='pockets')
    pocket_obj.pocket_search()

## End of Step 1
By now you should be left with a mirrored `output/` tree (one folder per `apo_structures`/`holo_structures`, project and replicate) holding everything Step 1 produced, e.g.:

```
output/apo_structures/apo8ITF/1/
├── dry_prot.pdb, dry_prot.xtc          # solvent/membrane-stripped trajectory (Step 1.1)
├── aligned_top.pdb, aligned_traj.xtc   # dry protein aligned to the reference structure (Step 1.1)
├── RMSD_protein_and_name_CA.csv        # per-group RMSD time series (Step 1.2)
├── RMSF_protein_and_name_CA_aligned_trj.csv
├── plots/                              # RMSD & RMSF figures (.pdf/.tiff)
└── pockets/                            # MDpocket/ATClus output (Step 1.3)
    ├── mdpout_dens_grid.dx, mdpout_dens_iso_3.pdb    # raw density grid & isosurface
    ├── mdpout_dens_iso_3-out-01.pdb ... -out-NN.pdb  # individual pockets, separated by ATClus
    ├── mdpout_dens_iso_3-out-01_descriptors.txt      # per-pocket volume/druggability descriptors
    └── mdpout_dens_iso_3-out-01_mdpocket_atoms.pdb   # atoms lining each pocket
```

Continue with **`step2_pocket_analysis`**, which reads the `pockets/` descriptor files and pocket PDBs above to assign global pocket IDs across replicates and genes, and to characterise them further.